In [1]:
# Importing Libraries
import pandas as pd
import sqlite3
import os

CLEAN_DIR = "../clean_data/"
DB_PATH = "../data/patents.db"

print("Libraries loaded")
print(f"Database will be saved at: {DB_PATH}")

Libraries loaded
Database will be saved at: ../data/patents.db


In [2]:
#Loading all clean CSV files
df_patents = pd.read_csv(CLEAN_DIR + "clean_patents.csv")
df_inventors = pd.read_csv(CLEAN_DIR + "clean_inventors.csv")
df_companies = pd.read_csv(CLEAN_DIR + "clean_companies.csv")
df_relationships = pd.read_csv(CLEAN_DIR + "clean_relationships.csv")

print(f"patents loaded:       {len(df_patents):,} rows")
print(f"inventors loaded:     {len(df_inventors):,} rows")
print(f"companies loaded:     {len(df_companies):,} rows")
print(f"relationships loaded: {len(df_relationships):,} rows")

patents loaded:       100,000 rows
inventors loaded:     100,000 rows
companies loaded:     98,987 rows
relationships loaded: 1,151 rows


In [3]:
# Creating database connection
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

print(f"Database created and connected")
print(f"Location: {DB_PATH}")

Database created and connected
Location: ../data/patents.db


In [4]:
# Creating tables schema
cursor.executescript("""
    DROP TABLE IF EXISTS patents;
    DROP TABLE IF EXISTS inventors;
    DROP TABLE IF EXISTS companies;
    DROP TABLE IF EXISTS relationships;

    CREATE TABLE patents (
        patent_id    TEXT PRIMARY KEY,
        patent_type  TEXT,
        filing_date  TEXT,
        year         INTEGER,
        title        TEXT,
        abstract     TEXT
    );

    CREATE TABLE inventors (
        inventor_id  TEXT,
        patent_id    TEXT,
        full_name    TEXT,
        country      TEXT
    );

    CREATE TABLE companies (
        company_id    TEXT,
        patent_id     TEXT,
        name          TEXT,
        assignee_type REAL
    );

    CREATE TABLE relationships (
        patent_id   TEXT,
        inventor_id TEXT,
        company_id  TEXT
    );
""")

conn.commit()
print("All 4 tables created successfully")

All 4 tables created successfully


In [5]:
# Loading Patents into database
df_patents[['patent_id','patent_type','filing_date','year','title','abstract']].to_sql(
    'patents', conn, if_exists='append', index=False
)

print(f"Patents loaded into database: {len(df_patents):,} rows")

Patents loaded into database: 100,000 rows


In [6]:
# Loading Inventors into database
df_inventors[['inventor_id','patent_id','full_name','country']].to_sql(
    'inventors', conn, if_exists='append', index=False
)

print(f"Inventors loaded into database: {len(df_inventors):,} rows")

Inventors loaded into database: 100,000 rows


In [7]:
# Loading Companies into database
df_companies[['company_id','patent_id','name','assignee_type']].to_sql(
    'companies', conn, if_exists='append', index=False
)

print(f"Companies loaded into database: {len(df_companies):,} rows")

Companies loaded into database: 98,987 rows


In [8]:
# Loading relationships into database
df_relationships[['patent_id','inventor_id','company_id']].to_sql(
    'relationships', conn, if_exists='append', index=False
)

print(f"Relationships loaded into database: {len(df_relationships):,} rows")

Relationships loaded into database: 1,151 rows


In [9]:
#Verifying all tables
tables = ['patents','inventors','companies','relationships']

for table in tables:
    count = cursor.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table}: {count:,} rows in database")

patents: 100,000 rows in database
inventors: 100,000 rows in database
companies: 98,987 rows in database
relationships: 1,151 rows in database


In [10]:
# Quick Test Query
print("Sample from patents table:")
result = pd.read_sql("SELECT patent_id, title, year FROM patents LIMIT 5", conn)
display(result)

print("\nSample from inventors table:")
result = pd.read_sql("SELECT inventor_id, full_name, country FROM inventors LIMIT 5", conn)
display(result)

Sample from patents table:


,patent_id,title,year
0,10000000,Coherent LADAR using intra-pixel quadrature de...,2018
1,10000001,Injection molding machine and mold thickness c...,2018
2,10000002,Method for manufacturing polymer film and co-e...,2018
3,10000003,Method for producing a container from a thermo...,2018
4,10000004,"Process of obtaining a double-oriented film, c...",2018



Sample from inventors table:


,inventor_id,full_name,country
0,fl:we_ln:jiang-165,Wenjing Jiang,CN
1,fl:ei_ln:baumker-1,Eiko BÄUMKER,DE
2,fl:ri_ln:kroeger-1,Richard Kroeger,None
3,fl:th_ln:bush-1,Thomas A. Bush,None
4,fl:ma_ln:boudreaux-4,Matthew F. Boudreaux,US


In [12]:
#Saving Schema SQL File
schema_sql = """
-- patents table
CREATE TABLE patents (
    patent_id    TEXT PRIMARY KEY,
    patent_type  TEXT,
    filing_date  TEXT,
    year         INTEGER,
    title        TEXT,
    abstract     TEXT
);

-- inventors table
CREATE TABLE inventors (
    inventor_id  TEXT,
    patent_id    TEXT,
    full_name    TEXT,
    country      TEXT
);

-- companies table
CREATE TABLE companies (
    company_id    TEXT,
    patent_id     TEXT,
    name          TEXT,
    assignee_type REAL
);

-- relationships table
CREATE TABLE relationships (
    patent_id   TEXT,
    inventor_id TEXT,
    company_id  TEXT
);
"""

with open("../sql/schema.sql", "w") as f:
    f.write(schema_sql)

print("schema.sql saved to sql/ folder")

schema.sql saved to sql/ folder


In [13]:
# Closing Connection
conn.close()
print("Database connection closed")



Database connection closed
